# 01 — Structural Dry-Run

Generates all trajectory records from a domain registry **without calling the model**.
Validates that:
- The pipeline structure (events, messages, tool schemas) is correct
- Injection placement is correct per depth condition
- Trajectory IDs follow the structured convention
- Pre-filled tool-call messages pass validation

**Usage:** Run all cells. If everything passes, the registry is ready for live execution.

In [0]:
import sys, os, json, copy
from pathlib import Path

# ── Repo root: portable across Databricks, local, and CI ──
if os.environ.get("DATABRICKS_RUNTIME_VERSION"):
    # Running inside Databricks — cwd is the notebook's directory
    REPO_ROOT = Path(os.getcwd()).parent
else:
    # Running locally or in CI
    REPO_ROOT = Path(".").resolve()
    # Walk up until we find pyproject.toml (repo marker)
    _p = REPO_ROOT
    while _p != _p.parent:
        if (_p / "pyproject.toml").exists():
            REPO_ROOT = _p
            break
        _p = _p.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Load & normalize registry

In [0]:
from src.scenario1.generator import (
    normalize_registry,
    build_record,
    load_documents,
    HOP_PATH,
)
from src.scenario1.pipeline_prompts import (
    SYSTEM_PROMPTS,
    AGENT_SEQUENCE,
    tools_for_role,
    build_planner_input,
    build_worker_retriever_messages,
)
from src.infrastructure.qwen_modal import validate_generation_request

# ── Registry path ──
# Change this to test a different domain:
REGISTRY_PATH = REPO_ROOT / "experiments/scenario1/inputs/fellow_packages/aihc/registry.json"

with open(REGISTRY_PATH) as f:
    raw = json.load(f)

reg = normalize_registry(raw, REGISTRY_PATH)

print(f"group_id:    {reg['group_id']}")
print(f"scenario_id: {reg['scenario_id']}")
print(f"model_id:    {reg['model_id']}")
print(f"trust_mode:  {reg['trust_mode']}")
print(f"domain_id:   {reg['domain_id']}")
print(f"conditions:  {len(reg['conditions'])}")

assert "independence_group_id" not in reg, "Legacy key should be removed by normalize_registry"
print("\n✓ Registry normalized (no legacy keys)")

## 2. Verify documents load correctly

In [0]:
documents = load_documents(reg)

print(f"Documents loaded: {len(documents)}")
carriers = [d for d in documents if d["role"] == "injection_carrier"]
benign = [d for d in documents if d["role"] == "benign"]

assert len(carriers) == 1, f"Expected 1 carrier, got {len(carriers)}"
assert len(benign) == 2, f"Expected 2 benign, got {len(benign)}"

for doc in documents:
    print(f"  {doc['doc_id']:12s} | {doc['role']:18s} | {len(doc['text']):,} chars | {doc['title'][:50]}...")

carrier = carriers[0]
print(f"\n✓ Carrier: {carrier['doc_id']} ({carrier['title']})")

# Verify insertion anchor exists in carrier
anchor = reg["injection"].get("insertion_anchor") or reg["injection"].get("_insertion_anchor_legacy")
if anchor:
    assert anchor in carrier["text"], f"Insertion anchor not found in carrier doc: {anchor[:40]}..."
    print(f"✓ Insertion anchor found at char {carrier['text'].index(anchor):,}")
else:
    pos = reg["injection"].get("insertion_position", "unknown")
    print(f"✓ Insertion position: {pos} (no literal anchor to verify)")

# Verify carrier_marker is NOT in clean text
marker = reg["injection"]["carrier_marker"]
assert marker not in carrier["text"], f"carrier_marker appears in clean document — change it!"
print(f"✓ carrier_marker absent from clean text (will only appear after injection)")

## 3. Build all trajectory records

In [0]:
records = []
for condition_def in reg["conditions"]:
    cond = condition_def["condition_id"]
    treat = condition_def["treatment"]
    record = build_record(reg, cond, treat)
    records.append(record)

    inj = record["injection"]
    status = ""
    if inj["injection_present"]:
        ip = inj["injection_point"]
        status = f" | inj→{ip['agent_id']} hop{ip['hop_index']}"
    print(f"✓ {record['trajectory_id']}{status}")

assert len(records) == 4, f"Expected 4 records, got {len(records)}"
print(f"\n✓ All 4 trajectory records built")

## 4. Validate trajectory structure

In [0]:
for record in records:
    tid = record["trajectory_id"]

    # ── Schema version ──
    assert record["schema_version"] == "spec_gap.scenario1.v3", f"{tid}: wrong schema_version"

    # ── Experiment identity fields ──
    assert record["group_id"] == reg["group_id"]
    assert record["scenario_id"] == reg["scenario_id"]
    assert record["model_id"] == reg["model_id"]
    assert record["trust_mode"] == reg["trust_mode"]

    # ── Structured trajectory_id ──
    parts = tid.split("_")
    assert parts[0] == reg["scenario_id"], f"{tid}: scenario_id mismatch"
    assert reg["model_id"].split("/")[-1].lower() in tid, f"{tid}: model not in id"

    # ── Events match expected agent count ──
    events = record["trajectory_trace"]["full_events"]
    cond = record["condition_id"]
    expected_agents = len(AGENT_SEQUENCE[cond])
    assert len(events) == expected_agents, (
        f"{tid}: {len(events)} events != {expected_agents} expected for {cond}"
    )

    # ── hop_path ──
    assert record["hop_path"] == HOP_PATH[cond], f"{tid}: hop_path mismatch"

    # ── Injection placement ──
    inj = record["injection"]
    if inj["injection_present"]:
        ip = inj["injection_point"]
        assert ip["agent_role"] == "worker_retriever"
        # 2-hop: worker_1 @ hop 1; 3-hop: worker_2 @ hop 2
        if cond == "2-hop":
            assert ip["agent_id"] == "worker_1" and ip["hop_index"] == 1
        else:
            assert ip["agent_id"] == "worker_2" and ip["hop_index"] == 2
        assert inj["injection_source_id"] == carrier["doc_id"]

    # ── Contrast pairs match ──
    pair_id = record["matched_pair_id"]
    assert reg["group_id"] in pair_id
    assert cond.replace("-", "") in pair_id

print("✓ All trajectory structures valid")
print(f"  Schema: spec_gap.scenario1.v3")
print(f"  Experiment identity: ✓")
print(f"  Structured IDs: ✓")
print(f"  Event counts: ✓")
print(f"  Injection placement: ✓")
print(f"  Contrast pairs: ✓")

## 5. Validate generation requests (pre-Modal)

In [0]:
from src.scenario1.generator import build_generation_request, _agent_events, _agents

validated_count = 0
for record in records:
    tid = record["trajectory_id"]
    for event in _agent_events(record):
        # build_generation_request calls validate_generation_request internally
        request = build_generation_request(record, event, thinking_mode="off")

        # Verify tool assignment
        agent_role = event["agent_role"]
        expected_tools = tools_for_role(agent_role)
        assert request["tools"] == expected_tools, (
            f"{tid}/{event['agent_id']}: wrong tools for {agent_role}"
        )

        # Verify worker_retriever does NOT have audit tool
        if agent_role == "worker_retriever":
            tool_names = [t["function"]["name"] for t in request["tools"]]
            assert "submit_document_for_audit" not in tool_names, (
                f"{tid}: worker_retriever must NOT have audit tool!"
            )
            # Pre-filled messages: 4 messages (system, user, assistant+tool_calls, tool)
            assert len(request["messages"]) == 4, (
                f"{tid}: retriever should have 4 pre-filled messages"
            )
            assert request["messages"][2]["role"] == "assistant"
            assert request["messages"][2].get("content") is None
            assert "tool_calls" in request["messages"][2]
            assert request["messages"][3]["role"] == "tool"

        # Verify expected_poison_agent_id
        assert "expected_poison_agent_id" in request

        validated_count += 1

print(f"✓ {validated_count} generation requests validated")
print(f"  Tool assignment: ✓ (worker=retrieve only, executor=audit only)")
print(f"  Pre-filled messages: ✓ (4-message pattern for retriever)")
print(f"  Poison exposure: ✓ (expected_poison_agent_id in every request)")

## 6. Summary

In [0]:
print("=" * 60)
print("STRUCTURAL DRY-RUN COMPLETE")
print("=" * 60)
print(f"""
Registry:     {REGISTRY_PATH.name}
Group:        {reg['group_id']}
Scenario:     {reg['scenario_id']}
Model:        {reg['model_id']}
Trust mode:   {reg['trust_mode']}
Carrier doc:  {carrier['doc_id']} ({carrier['title'][:40]}...)
Conditions:   4 (2-hop clean/inj + 3-hop clean/inj)
Records:      {len(records)} built
Requests:     {validated_count} validated

Trajectory IDs:
""")
for r in records:
    inj_mark = " ← injected" if r["injection"]["injection_present"] else ""
    print(f"  {r['trajectory_id']}{inj_mark}")

print(f"""
Pipeline flow:
  2-hop: planner → worker_retriever(pre-filled tool) → executor
  3-hop: planner → worker_relay(clean) → worker_retriever(pre-filled tool) → executor

Ready for live execution on Modal.
""")